# NALAPRO — 20 Newsgroups Text Classification
**Apple Silicon MacBook** — MPS GPU backend

| Part | What we do |
|------|------------|
| 0 | Imports & configuration |
| Data | Load & clean the dataset |
| 1 | Word2Vec · TF-IDF · FastText → 2-layer NN |
| 2 | BERT direct fine-tuning |
| 3 | BERT MLM domain adaptation → classification |
| 4 | Llama-3.2-1B zero-shot & few-shot |
| 5 | Final comparison chart |

Each section is **independent** — re-run any cell without redoing the whole notebook.

WANDB URL: https://wandb.ai/talhamch-hslu/nalapro-20newsgroups/runs/uxxqvjz6

(WANDB Wont allow me to make my run public so im sharing a URL of report)

WANDB REPORT URL:  https://api.wandb.ai/links/talhamch-hslu/6wizrh10

Tools Used: Claude AI

## 0 — Install & Imports
Run once per environment.

In [12]:
!pip install nbformat --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [nbformat]


In [13]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "torch", "transformers", "gensim", "nltk",
    "matplotlib", "scikit-learn", "accelerate",
    "wandb",        # Weights & Biases — experiment tracking
], check=False)
print("Packages installed.")


Packages installed.


In [14]:
import os
import wandb

# Set your Weights & Biases API key environment variable
os.environ["WANDB_API_KEY"] = "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"

# Log into wandb using the environment variable above
wandb.login()

True

In [15]:
import os, re, gc, json, random, warnings
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim import AdamW

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.decomposition import PCA

from gensim.models import Word2Vec, FastText
from nltk.tokenize import word_tokenize
import nltk

from transformers import (
    BertTokenizerFast, BertForSequenceClassification, BertForMaskedLM,
    DataCollatorForLanguageModeling, AutoTokenizer, AutoModelForCausalLM,
    get_linear_schedule_with_warmup, pipeline as hf_pipeline,
)

import wandb  # Weights & Biases — tracks every metric, plot and config automatically

warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Device: MPS → CUDA → CPU ──────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('Device: MPS (Apple Silicon GPU)')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'Device: CUDA — {torch.cuda.get_device_name(0)}')
else:
    DEVICE = torch.device('cpu')
    print('Device: CPU')

def free_memory():
    gc.collect()
    if DEVICE.type == 'mps':   torch.mps.empty_cache()
    elif DEVICE.type == 'cuda': torch.cuda.empty_cache()

# ── Hyperparameters ───────────────────────────────────────────────────────────
NUM_CLASSES   = 20
EMBEDDING_DIM = 100
W2V_FEW       = 1
W2V_MANY      = 15
NN_EPOCHS     = 10
BATCH_SIZE    = 64
LR_NN         = 1e-3
HIDDEN_SIZE   = 256

BERT_MODEL    = 'bert-base-uncased'
MAX_SEQ_LEN   = 128
BERT_BATCH    = 16
BERT_EPOCHS   = 3
BERT_LR       = 2e-5
WARMUP_RATIO  = 0.1
BERT_SUBSET   = 3000
MLM_EPOCHS    = 1
MLM_LR        = 5e-5
MLM_MASK_PROB = 0.15

LLAMA_MODEL       = 'meta-llama/Llama-3.2-1B-Instruct'
EVAL_SIZE         = 60
FEW_SHOT_KS       = [1, 3, 5]
MAX_NEW_TOKENS    = 15
FEW_SHOT_EX_CHARS = 80

RESULTS        = {}
BERT_TOKENIZER = None

# ── Weights & Biases — single project, one run per Part ──────────────────────
# wandb.login() reads WANDB_API_KEY from the environment automatically.
# To set it: export WANDB_API_KEY=<your_key>   (or paste it when prompted)
wandb.login()

WANDB_PROJECT = "nalapro-20newsgroups"  # all Parts log to the same W&B project

# Shared hyperparameter config logged to every run so you can compare them
WANDB_CONFIG = {
    "seed":           SEED,
    "device":         str(DEVICE),
    "num_classes":    NUM_CLASSES,
    "embedding_dim":  EMBEDDING_DIM,
    "w2v_few_epochs": W2V_FEW,
    "w2v_many_epochs":W2V_MANY,
    "nn_epochs":      NN_EPOCHS,
    "nn_batch_size":  BATCH_SIZE,
    "nn_lr":          LR_NN,
    "nn_hidden_size": HIDDEN_SIZE,
    "bert_model":     BERT_MODEL,
    "bert_max_seq":   MAX_SEQ_LEN,
    "bert_batch":     BERT_BATCH,
    "bert_epochs":    BERT_EPOCHS,
    "bert_lr":        BERT_LR,
    "bert_warmup":    WARMUP_RATIO,
    "bert_subset":    BERT_SUBSET,
    "mlm_epochs":     MLM_EPOCHS,
    "mlm_lr":         MLM_LR,
    "mlm_mask_prob":  MLM_MASK_PROB,
    "llama_model":    LLAMA_MODEL,
    "llama_eval_size":EVAL_SIZE,
}

print('Config ready. W&B project:', WANDB_PROJECT)


Device: MPS (Apple Silicon GPU)
Config ready. W&B project: nalapro-20newsgroups


---
## Data Loading
Run once. All parts share the `data` dict.

In [16]:
print('Loading 20 Newsgroups...')

# remove= strips metadata that would trivially reveal the category
ng = fetch_20newsgroups(subset='all', remove=('headers','footers','quotes'), random_state=SEED)
raw_texts, raw_labels, categories = ng.data, ng.target, ng.target_names
print(f'{len(raw_texts)} documents | {NUM_CLASSES} categories')
for i, c in enumerate(categories): print(f'  {i:2d}. {c}')

# Three cleaning strategies for three different model types
def clean_classic(text):            # letters only — for Word2Vec / TF-IDF
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+|\S+@\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def clean_bert(text):               # keep punctuation — for BERT WordPiece
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+|\S+@\S+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()[:600]

def clean_llm(text):                # slightly longer context — for Llama
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+|\S+@\S+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()[:800]

print('Cleaning texts...')
cleaned_classic = [clean_classic(t) for t in raw_texts]
tokenised       = [word_tokenize(t) for t in cleaned_classic]
cleaned_bert    = [clean_bert(t)    for t in raw_texts]
cleaned_llm     = [clean_llm(t)     for t in raw_texts]

# Stratified 80/20 split — same split reused for ALL experiments
idx_all = list(range(len(raw_texts)))
train_idx, test_idx, train_labels, test_labels = train_test_split(
    idx_all, raw_labels, test_size=0.2, random_state=SEED, stratify=raw_labels)

# Smaller BERT subset to keep training time manageable
sub_train_idx, _, sub_train_labels, _ = train_test_split(
    train_idx, train_labels, train_size=BERT_SUBSET, random_state=SEED, stratify=train_labels)

data = dict(
    raw_texts=raw_texts, categories=categories,
    cleaned_classic=cleaned_classic, tokenised=tokenised,
    cleaned_bert=cleaned_bert, cleaned_llm=cleaned_llm,
    train_idx=train_idx, test_idx=test_idx,
    train_labels=train_labels, test_labels=test_labels,
    sub_train_idx=sub_train_idx, sub_train_labels=sub_train_labels,
)
print(f'Full split  — train: {len(train_idx)}, test: {len(test_idx)}')
print(f'BERT subset — train: {len(sub_train_idx)}, test: {len(test_idx)}')


Loading 20 Newsgroups...
18846 documents | 20 categories
   0. alt.atheism
   1. comp.graphics
   2. comp.os.ms-windows.misc
   3. comp.sys.ibm.pc.hardware
   4. comp.sys.mac.hardware
   5. comp.windows.x
   6. misc.forsale
   7. rec.autos
   8. rec.motorcycles
   9. rec.sport.baseball
  10. rec.sport.hockey
  11. sci.crypt
  12. sci.electronics
  13. sci.med
  14. sci.space
  15. soc.religion.christian
  16. talk.politics.guns
  17. talk.politics.mideast
  18. talk.politics.misc
  19. talk.religion.misc
Cleaning texts...
Full split  — train: 15076, test: 3770
BERT subset — train: 3000, test: 3770


---
## Part 1 — Classical Embeddings + 2-Layer Neural Network
```
Input → Linear(→256) → ReLU → Dropout(0.3) → Linear(→20) → Logits
```
The **same network** is tested with four different vectorisations.

### Part 1 Helpers

In [17]:
class SimpleNN(nn.Module):
    """2-layer feedforward classifier: Linear -> ReLU -> Dropout -> Linear"""
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, HIDDEN_SIZE),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(HIDDEN_SIZE, NUM_CLASSES),
        )
    def forward(self, x): return self.net(x)


def train_nn(X_train, y_train, X_test, y_test, label):
    """Train SimpleNN, log every epoch to W&B, return (test_accuracy, epoch_losses)."""
    X_tr = torch.tensor(X_train, dtype=torch.float32)
    y_tr = torch.tensor(np.array(y_train), dtype=torch.long)
    X_te = torch.tensor(X_test,  dtype=torch.float32)
    y_te = torch.tensor(np.array(y_test),  dtype=torch.long)

    loader    = DataLoader(TensorDataset(X_tr, y_tr),
                           batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    model     = SimpleNN(X_train.shape[1]).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR_NN)
    losses    = []

    # Start a W&B run for this vectorisation experiment.
    # name=label makes each bar in the W&B comparison chart clearly labelled.
    run = wandb.init(
        project=WANDB_PROJECT,
        name=f"Part1-{label}",
        group="Part1-SimpleNN",          # groups all Part-1 runs together in W&B
        config={**WANDB_CONFIG,
                "vectoriser": label,
                "input_dim":  X_train.shape[1]},
        reinit=True,                     # allow multiple runs in the same kernel session
    )

    for epoch in range(NN_EPOCHS):
        model.train()
        total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item()
        avg = total / len(loader)
        losses.append(avg)
        print(f'  [{label}] Epoch {epoch+1}/{NN_EPOCHS}  Loss: {avg:.4f}')

        # Log epoch-level training loss to W&B
        wandb.log({"train/loss": avg, "epoch": epoch + 1})

    model.eval()
    with torch.no_grad():
        preds = model(X_te.to(DEVICE)).argmax(dim=1).cpu().numpy()
    acc = accuracy_score(y_te.numpy(), preds)
    print(f'  [{label}] Test Accuracy: {acc*100:.2f}%\n')

    # Log final test accuracy and a summary metric (shown in W&B run overview table)
    wandb.log({"test/accuracy": acc})
    wandb.summary["test_accuracy"] = acc

    run.finish()   # close this W&B run cleanly before the next one starts
    return acc, losses


def doc_vector(tokens, model, dim):
    """Mean-pool word vectors to get a single document vector."""
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

print('Part 1 helpers ready.')


Part 1 helpers ready.


### Part 1B — Word2Vec: 1 epoch vs 15 epochs
**1 epoch** = near-random vectors → poor accuracy.  
**15 epochs** = co-occurrence statistics converge → semantically meaningful space.

In [18]:
tokenised    = data['tokenised']
train_idx    = data['train_idx']
test_idx     = data['test_idx']
train_labels = data['train_labels']
test_labels  = data['test_labels']

print(f'Training Word2Vec — {W2V_FEW} epoch (undertrained)...')
w2v_few = Word2Vec(sentences=tokenised, vector_size=EMBEDDING_DIM,
                   window=5, min_count=2, workers=4, epochs=W2V_FEW, seed=SEED)

print(f'Training Word2Vec — {W2V_MANY} epochs (converged)...')
w2v_many = Word2Vec(sentences=tokenised, vector_size=EMBEDDING_DIM,
                    window=5, min_count=2, workers=4, epochs=W2V_MANY, seed=SEED)

X_few  = np.array([doc_vector(tokenised[i], w2v_few,  EMBEDDING_DIM) for i in range(len(tokenised))])
X_many = np.array([doc_vector(tokenised[i], w2v_many, EMBEDDING_DIM) for i in range(len(tokenised))])
print(f'Document matrix shape: {X_few.shape}')


Training Word2Vec — 1 epoch (undertrained)...
Training Word2Vec — 15 epochs (converged)...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Document matrix shape: (18846, 100)


In [19]:
# Joint PCA: fit on both models' vectors together so both panels share the same axes
sample_words = ['computer','god','gun','space','game','christian','health','science','car','hockey']
sample_words = [w for w in sample_words if w in w2v_few.wv and w in w2v_many.wv]

vf = np.array([w2v_few.wv[w]  for w in sample_words])
vm = np.array([w2v_many.wv[w] for w in sample_words])

pca   = PCA(n_components=2, random_state=SEED)
all2d = pca.fit_transform(np.vstack([vf, vm]))
n     = len(sample_words)
pf, pm = all2d[:n], all2d[n:]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, pts, title in zip(axes, [pf, pm],
    [f'Word2Vec — {W2V_FEW} epoch (undertrained)', f'Word2Vec — {W2V_MANY} epochs (converged)']):
    ax.scatter(pts[:,0], pts[:,1], c='steelblue', s=100, zorder=3)
    for i, w in enumerate(sample_words):
        ax.annotate(w, (pts[i,0], pts[i,1]), fontsize=9, ha='right',
                    xytext=(-4,4), textcoords='offset points')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('PCA dim 1'); ax.set_ylabel('PCA dim 2')
    ax.grid(True, linestyle='--', alpha=0.4)
plt.suptitle('Word Vectors: 1 epoch vs 15 epochs (joint PCA)', fontsize=13)
plt.tight_layout(); plt.savefig('word2vec_pca.png', dpi=150); plt.show(); plt.close()

# Log the PCA plot to W&B so it is permanently saved in the project media gallery.
# We open a brief throwaway run just for the visualisation artifact.
with wandb.init(project=WANDB_PROJECT, name="Part1-W2V-PCA",
                group="Part1-SimpleNN", config=WANDB_CONFIG, reinit=True):
    wandb.log({"word2vec_pca": wandb.Image("word2vec_pca.png",
               caption="PCA of word vectors: 1 epoch vs 15 epochs")})

print(f'Euclidean distance per word (1ep vs {W2V_MANY}ep):')
distances = {}
for i, w in enumerate(sample_words):
    d = float(np.linalg.norm(vf[i]-vm[i]))
    distances[w] = d
    print(f'  {w:15s}: {d:.4f}')


Euclidean distance per word (1ep vs 15ep):
  computer       : 13.4076
  god            : 23.2640
  gun            : 18.5208
  space          : 16.8309
  game           : 17.4286
  christian      : 15.4206
  health         : 16.8799
  science        : 14.5274
  car            : 18.0421
  hockey         : 14.5265


In [20]:
print('NN on Word2Vec 1 epoch...')
acc_w2v_few,  _ = train_nn(X_few[train_idx],  train_labels, X_few[test_idx],  test_labels, 'W2V-1ep')

print(f'NN on Word2Vec {W2V_MANY} epochs...')
acc_w2v_many, _ = train_nn(X_many[train_idx], train_labels, X_many[test_idx], test_labels, 'W2V-15ep')

RESULTS['W2V 1ep']  = acc_w2v_few
RESULTS['W2V 15ep'] = acc_w2v_many


NN on Word2Vec 1 epoch...


  [W2V-1ep] Epoch 1/10  Loss: 2.7810
  [W2V-1ep] Epoch 2/10  Loss: 2.5943
  [W2V-1ep] Epoch 3/10  Loss: 2.5377
  [W2V-1ep] Epoch 4/10  Loss: 2.5134
  [W2V-1ep] Epoch 5/10  Loss: 2.4872
  [W2V-1ep] Epoch 6/10  Loss: 2.4701
  [W2V-1ep] Epoch 7/10  Loss: 2.4570
  [W2V-1ep] Epoch 8/10  Loss: 2.4474
  [W2V-1ep] Epoch 9/10  Loss: 2.4359
  [W2V-1ep] Epoch 10/10  Loss: 2.4258
  [W2V-1ep] Test Accuracy: 24.64%



epoch,▁▂▃▃▄▅▆▆▇█
test/accuracy,▁
train/loss,█▄▃▃▂▂▂▁▁▁
epoch,10
test/accuracy,0.24642
test_accuracy,0.24642
train/loss,2.4258


NN on Word2Vec 15 epochs...


  [W2V-15ep] Epoch 1/10  Loss: 2.1708
  [W2V-15ep] Epoch 2/10  Loss: 1.6361
  [W2V-15ep] Epoch 3/10  Loss: 1.5054
  [W2V-15ep] Epoch 4/10  Loss: 1.4425
  [W2V-15ep] Epoch 5/10  Loss: 1.3957
  [W2V-15ep] Epoch 6/10  Loss: 1.3690
  [W2V-15ep] Epoch 7/10  Loss: 1.3426
  [W2V-15ep] Epoch 8/10  Loss: 1.3134
  [W2V-15ep] Epoch 9/10  Loss: 1.3008
  [W2V-15ep] Epoch 10/10  Loss: 1.2887
  [W2V-15ep] Test Accuracy: 57.93%



epoch,▁▂▃▃▄▅▆▆▇█
test/accuracy,▁
train/loss,█▄▃▂▂▂▁▁▁▁
epoch,10
test/accuracy,0.57931
test_accuracy,0.57931
train/loss,1.28869


### Part 1C — TF-IDF
Fit **only on training data** — no vocabulary leakage to test set.

In [21]:
classic = data['cleaned_classic']

tfidf      = TfidfVectorizer(max_features=10_000, sublinear_tf=True, min_df=2)
X_tr_tfidf = tfidf.fit_transform([classic[i] for i in train_idx]).toarray()
X_te_tfidf = tfidf.transform(   [classic[i] for i in test_idx]).toarray()
print(f'TF-IDF matrix — train: {X_tr_tfidf.shape}, test: {X_te_tfidf.shape}')

acc_tfidf, _ = train_nn(X_tr_tfidf, train_labels, X_te_tfidf, test_labels, 'TF-IDF')
RESULTS['TF-IDF'] = acc_tfidf


TF-IDF matrix — train: (15076, 10000), test: (3770, 10000)


  [TF-IDF] Epoch 1/10  Loss: 2.2514
  [TF-IDF] Epoch 2/10  Loss: 1.0151
  [TF-IDF] Epoch 3/10  Loss: 0.6354
  [TF-IDF] Epoch 4/10  Loss: 0.4323
  [TF-IDF] Epoch 5/10  Loss: 0.3101
  [TF-IDF] Epoch 6/10  Loss: 0.2328
  [TF-IDF] Epoch 7/10  Loss: 0.1855
  [TF-IDF] Epoch 8/10  Loss: 0.1551
  [TF-IDF] Epoch 9/10  Loss: 0.1354
  [TF-IDF] Epoch 10/10  Loss: 0.1226
  [TF-IDF] Test Accuracy: 74.99%



epoch,▁▂▃▃▄▅▆▆▇█
test/accuracy,▁
train/loss,█▄▃▂▂▁▁▁▁▁
epoch,10
test/accuracy,0.74987
test_accuracy,0.74987
train/loss,0.12265


### Part 1D — FastText (improvement experiment)
FastText uses character n-grams → handles OOV words and typos better than Word2Vec.

In [22]:
print(f'Training FastText — {W2V_MANY} epochs...')
ft   = FastText(sentences=tokenised, vector_size=EMBEDDING_DIM,
                window=5, min_count=2, workers=4, epochs=W2V_MANY, seed=SEED)
X_ft = np.array([doc_vector(tokenised[i], ft, EMBEDDING_DIM) for i in range(len(tokenised))])

acc_ft, _ = train_nn(X_ft[train_idx], train_labels, X_ft[test_idx], test_labels, 'FastText')
RESULTS['FastText'] = acc_ft

# Part 1 summary bar chart
names  = [f'W2V\n{W2V_FEW}ep', f'W2V\n{W2V_MANY}ep', 'TF-IDF', 'FastText']
accs   = [acc_w2v_few, acc_w2v_many, acc_tfidf, acc_ft]
colors = ['#5e81ac','#81a1c1','#a3be8c','#ebcb8b']
fig, ax = plt.subplots(figsize=(9,5))
bars = ax.bar(names, accs, color=colors, edgecolor='black', width=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{acc*100:.2f}%', ha='center', va='bottom', fontsize=11)
ax.set_ylim(0,1); ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Part 1 — Classical Embeddings + 2-Layer NN', fontsize=13)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout(); plt.savefig('part1_results.png', dpi=150); plt.show(); plt.close()

# Log the Part 1 summary bar chart and a grouped accuracy table to W&B
with wandb.init(project=WANDB_PROJECT, name="Part1-Summary",
                group="Part1-SimpleNN", config=WANDB_CONFIG, reinit=True):
    wandb.log({
        "part1_comparison_chart": wandb.Image("part1_results.png",
                                              caption="Part 1 — all vectorisers"),
        # Bar chart as a W&B BarChart so results appear in the W&B dashboard
        "part1_accuracies": wandb.plot.bar(
            wandb.Table(columns=["Method", "Accuracy (%)"],
                        data=[[n.replace("\n"," "), round(a*100,2)] for n,a in zip(names,accs)]),
            "Method", "Accuracy (%)", title="Part 1 Accuracy Comparison"
        ),
    })

print('Part 1 complete.')


Training FastText — 15 epochs...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


  [FastText] Epoch 1/10  Loss: 2.3481
  [FastText] Epoch 2/10  Loss: 1.8594
  [FastText] Epoch 3/10  Loss: 1.6874
  [FastText] Epoch 4/10  Loss: 1.6027
  [FastText] Epoch 5/10  Loss: 1.5435
  [FastText] Epoch 6/10  Loss: 1.4893
  [FastText] Epoch 7/10  Loss: 1.4676
  [FastText] Epoch 8/10  Loss: 1.4371
  [FastText] Epoch 9/10  Loss: 1.4188
  [FastText] Epoch 10/10  Loss: 1.3988
  [FastText] Test Accuracy: 54.75%



epoch,▁▂▃▃▄▅▆▆▇█
test/accuracy,▁
train/loss,█▄▃▃▂▂▂▁▁▁
epoch,10
test/accuracy,0.54748
test_accuracy,0.54748
train/loss,1.39879


Part 1 complete.


---
## Part 2 — BERT Direct Fine-tuning
Fine-tune `bert-base-uncased` (110M params) on the 20-class classification task.

### BERT Helpers (shared with Part 3)

In [ ]:
BERT_TOKENIZER = BertTokenizerFast.from_pretrained(BERT_MODEL)
print(f'Tokenizer vocab: {BERT_TOKENIZER.vocab_size:,}')


class TextDataset(Dataset):
    """Tokenises text on-the-fly for BERT. Returns input_ids, attention_mask, label."""
    def __init__(self, texts, labels):
        self.texts, self.labels = texts, labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = BERT_TOKENIZER(str(self.texts[idx]), max_length=MAX_SEQ_LEN,
                              padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label': torch.tensor(int(self.labels[idx]), dtype=torch.long)}


def make_loaders(train_texts, train_labels, test_texts, test_labels):
    # num_workers=0 and pin_memory=False required for MPS compatibility
    return (
        DataLoader(TextDataset(train_texts, train_labels),
                   batch_size=BERT_BATCH, shuffle=True,  num_workers=0, pin_memory=False),
        DataLoader(TextDataset(test_texts,  test_labels),
                   batch_size=BERT_BATCH, shuffle=False, num_workers=0, pin_memory=False),
    )


def bert_train_eval(model, train_loader, test_loader, epochs, lr, label,
                    wandb_run_name=None, wandb_group=None, extra_config=None):
    """AdamW + linear warmup + gradient clipping + W&B logging.

    New parameters vs the original:
        wandb_run_name  – display name of the W&B run  (e.g. "Part2-BERT")
        wandb_group     – W&B group to file it under   (e.g. "Part2-BERT")
        extra_config    – dict merged into WANDB_CONFIG for this run
    """
    model      = model.to(DEVICE)
    optimizer  = AdamW(model.parameters(), lr=lr, eps=1e-8, weight_decay=0.01)
    total_steps= len(train_loader) * epochs
    scheduler  = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps*WARMUP_RATIO),
        num_training_steps=total_steps)
    criterion  = nn.CrossEntropyLoss()
    epoch_losses, epoch_accs = [], []
    all_preds, all_trues = [], []

    # Start the W&B run for this BERT experiment
    run_cfg = {**WANDB_CONFIG, "bert_lr": lr, "bert_epochs": epochs}
    if extra_config:
        run_cfg.update(extra_config)

    run = wandb.init(
        project=WANDB_PROJECT,
        name=wandb_run_name or f"BERT-{label}",
        group=wandb_group or label,
        config=run_cfg,
        reinit=True,
    )
    # Watch the model: log gradients and parameter histograms every 100 steps.
    # This lets you see if any layer's gradients vanish or explode.
    wandb.watch(model, log="gradients", log_freq=100)

    global_step = 0   # counts total gradient update steps across all epochs

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(train_loader):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(input_ids=ids, attention_mask=mask).logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()
            global_step += 1

            # Log every-step training loss so you can see the loss curve in real time
            wandb.log({"train/step_loss": loss.item(),
                       "train/lr": scheduler.get_last_lr()[0],
                       "step": global_step})

            if (step+1) % 50 == 0:
                print(f'  [{label}] Epoch {epoch+1} Step {step+1}/{len(train_loader)} Loss: {loss.item():.4f}')

        avg_loss = total_loss / len(train_loader)
        epoch_losses.append(avg_loss)

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                out = model(input_ids=batch['input_ids'].to(DEVICE),
                            attention_mask=batch['attention_mask'].to(DEVICE))
                preds.extend(out.logits.argmax(dim=1).cpu().numpy())
                trues.extend(batch['label'].numpy())
        acc = accuracy_score(trues, preds)
        epoch_accs.append(acc)
        all_preds, all_trues = preds, trues

        # Log epoch-level loss and accuracy together
        wandb.log({"train/epoch_loss": avg_loss,
                   "eval/accuracy":    acc,
                   "epoch":            epoch + 1})
        print(f'\n  [{label}] Epoch {epoch+1}/{epochs} — Loss: {avg_loss:.4f}  Acc: {acc*100:.2f}%\n')

    final_acc = accuracy_score(all_trues, all_preds)

    # Log a per-class confusion matrix to W&B so you can see which categories are confused
    #wandb.log({
       # "eval/confusion_matrix": wandb.plot.confusion_matrix(
         #   probs=None,
         #   y_true=all_trues,
         #   preds=all_preds,
         #   class_names=list(data['categories']),
       # )
    })
    wandb.summary["final_test_accuracy"] = final_acc
    run.finish()

    free_memory()
    return final_acc, epoch_losses, epoch_accs, all_trues, all_preds


print('BERT helpers ready.')


Tokenizer vocab: 30,522
BERT helpers ready.


In [24]:
train_texts = [data['cleaned_bert'][i] for i in data['sub_train_idx']]
test_texts  = [data['cleaned_bert'][i] for i in data['test_idx']]
train_loader, test_loader = make_loaders(
    train_texts, data['sub_train_labels'], test_texts, data['test_labels'])
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

bert_cls = BertForSequenceClassification.from_pretrained(
    BERT_MODEL, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True)
print(f'Trainable params: {sum(p.numel() for p in bert_cls.parameters() if p.requires_grad):,}')

acc_bert, bert_losses, bert_accs, bert_true, bert_preds = bert_train_eval(
    bert_cls, train_loader, test_loader, BERT_EPOCHS, BERT_LR, 'BERT',
    wandb_run_name="Part2-BERT-DirectFinetune",
    wandb_group="Part2-BERT",
    extra_config={"phase": "direct_finetune", "bert_subset": BERT_SUBSET},
)

RESULTS['BERT (Part 2)'] = acc_bert
print(f'Final BERT Accuracy: {acc_bert*100:.2f}%')
print(classification_report(bert_true, bert_preds, target_names=data['categories']))

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13,4))
axes[0].plot(range(1,BERT_EPOCHS+1), bert_losses, marker='o', color='#5e81ac', linewidth=2)
axes[0].set_title('BERT — Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True, linestyle='--', alpha=0.5)
axes[1].plot(range(1,BERT_EPOCHS+1), [a*100 for a in bert_accs], marker='s', color='#a3be8c', linewidth=2)
axes[1].set_title('BERT — Test Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].grid(True, linestyle='--', alpha=0.5)
plt.suptitle('Part 2 — BERT Direct Fine-tuning', fontsize=13)
plt.tight_layout(); plt.savefig('part2_bert.png', dpi=150); plt.show(); plt.close()

# Upload the training-curves plot to W&B as a permanent image artifact
with wandb.init(project=WANDB_PROJECT, name="Part2-TrainingCurves",
                group="Part2-BERT", config=WANDB_CONFIG, reinit=True):
    wandb.log({"part2_training_curves": wandb.Image("part2_bert.png",
               caption="Part 2 BERT loss and accuracy curves")})

bert_cls.save_pretrained('bert_cls_checkpoint')
BERT_TOKENIZER.save_pretrained('bert_cls_checkpoint')
print('Saved: bert_cls_checkpoint/ | Part 2 complete.')


Train batches: 188 | Test batches: 236


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8514.49it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

Trainable params: 109,497,620


  [BERT] Epoch 1 Step 50/188 Loss: 2.9172
  [BERT] Epoch 1 Step 100/188 Loss: 2.5419
  [BERT] Epoch 1 Step 150/188 Loss: 2.1293

  [BERT] Epoch 1/3 — Loss: 2.5223  Acc: 54.62%

  [BERT] Epoch 2 Step 50/188 Loss: 1.3980
  [BERT] Epoch 2 Step 100/188 Loss: 1.6994
  [BERT] Epoch 2 Step 150/188 Loss: 1.8397

  [BERT] Epoch 2/3 — Loss: 1.5442  Acc: 62.47%

  [BERT] Epoch 3 Step 50/188 Loss: 0.8939
  [BERT] Epoch 3 Step 100/188 Loss: 1.0789
  [BERT] Epoch 3 Step 150/188 Loss: 1.3509

  [BERT] Epoch 3/3 — Loss: 1.1849  Acc: 65.46%



epoch,▁▅█
eval/accuracy,▁▆█
step,▁▁▁▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇███
train/epoch_loss,█▃▁
train/lr,▄▄▇████▇▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁
train/step_loss,████▆▆▅▆▄▄▄▄▄▄▃▃▃▄▃▃▃▂▃▁▂▃▃▄▂▃▂▃▃▁▂▂▂▁▃▄
epoch,3
eval/accuracy,0.65464
final_test_accuracy,0.65464
step,564
train/epoch_loss,1.18485


Final BERT Accuracy: 65.46%
                          precision    recall  f1-score   support

             alt.atheism       0.36      0.26      0.30       160
           comp.graphics       0.77      0.49      0.60       195
 comp.os.ms-windows.misc       0.62      0.47      0.54       197
comp.sys.ibm.pc.hardware       0.51      0.63      0.56       196
   comp.sys.mac.hardware       0.70      0.40      0.51       193
          comp.windows.x       0.63      0.88      0.73       198
            misc.forsale       0.75      0.77      0.76       195
               rec.autos       0.47      0.77      0.59       198
         rec.motorcycles       0.73      0.61      0.66       199
      rec.sport.baseball       0.91      0.80      0.85       199
        rec.sport.hockey       0.86      0.90      0.88       200
               sci.crypt       0.72      0.69      0.70       198
         sci.electronics       0.63      0.68      0.65       197
                 sci.med       0.82      0.79  

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.81it/s]

Saved: bert_cls_checkpoint/ | Part 2 complete.


---
## Part 3 — BERT MLM Domain Adaptation → Classification
**Phase 1**: Continue BERT's masked language model training on newsgroup text (no labels needed).  
**Phase 2**: Add classification head and fine-tune.  
*Motivation*: BERT was pre-trained on Wikipedia. Newsgroups are informal and jargon-heavy. Even 1 MLM epoch shifts BERT's representations toward the target domain.

In [25]:
class MLMDataset(Dataset):
    """Returns input_ids and attention_mask only — labels are added by DataCollator."""
    def __init__(self, texts):
        self.texts = texts
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = BERT_TOKENIZER(str(self.texts[idx]), max_length=MAX_SEQ_LEN,
                              padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0)}


# ── Phase 1: MLM domain adaptation ───────────────────────────────────────────
mlm_texts  = [data['cleaned_bert'][i] for i in data['sub_train_idx']]
collator   = DataCollatorForLanguageModeling(
    tokenizer=BERT_TOKENIZER, mlm=True, mlm_probability=MLM_MASK_PROB)
mlm_loader = DataLoader(MLMDataset(mlm_texts), batch_size=BERT_BATCH,
                        shuffle=True, collate_fn=collator, num_workers=0, pin_memory=False)
print(f'MLM: {len(mlm_texts)} docs | {len(mlm_loader)} batches/epoch')

mlm_model = BertForMaskedLM.from_pretrained(BERT_MODEL).to(DEVICE)
mlm_opt   = AdamW(mlm_model.parameters(), lr=MLM_LR, eps=1e-8)
mlm_total = len(mlm_loader) * MLM_EPOCHS
mlm_sched = get_linear_schedule_with_warmup(
    mlm_opt, num_warmup_steps=int(mlm_total*WARMUP_RATIO), num_training_steps=mlm_total)

mlm_losses  = []
mlm_step    = 0

# Open a dedicated W&B run for the MLM pretraining phase
mlm_run = wandb.init(
    project=WANDB_PROJECT,
    name="Part3-MLM-DomainAdaptation",
    group="Part3-BERT-MLM",
    config={**WANDB_CONFIG,
            "phase":        "mlm_pretraining",
            "mlm_lr":       MLM_LR,
            "mlm_epochs":   MLM_EPOCHS,
            "mlm_mask_prob":MLM_MASK_PROB},
    reinit=True,
)

for epoch in range(MLM_EPOCHS):
    mlm_model.train(); total = 0.0
    for step, batch in enumerate(mlm_loader):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['labels'].to(DEVICE)
        mlm_opt.zero_grad()
        loss = mlm_model(input_ids=ids, attention_mask=mask, labels=lbls).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), 1.0)
        mlm_opt.step(); mlm_sched.step()
        total += loss.item()
        mlm_step += 1

        # Log every-step MLM loss so the pretraining curve is fully visible in W&B
        wandb.log({"mlm/step_loss": loss.item(),
                   "mlm/lr":        mlm_sched.get_last_lr()[0],
                   "mlm_step":      mlm_step})

        if (step+1) % 50 == 0:
            print(f'  [MLM] Epoch {epoch+1} Step {step+1}/{len(mlm_loader)} Loss: {loss.item():.4f}')
    avg = total / len(mlm_loader)
    mlm_losses.append(avg)
    wandb.log({"mlm/epoch_loss": avg, "mlm_epoch": epoch + 1})
    print(f'\n  [MLM] Epoch {epoch+1} avg loss: {avg:.4f}\n')

mlm_run.finish()  # close Phase 1 run

mlm_model.save_pretrained('bert_mlm_adapted')
BERT_TOKENIZER.save_pretrained('bert_mlm_adapted')
del mlm_model; free_memory()
print('Saved: bert_mlm_adapted/')

# ── Phase 2: Classification fine-tuning on adapted encoder ───────────────────
print('Loading MLM-adapted BERT for classification...')
bert_mlm_cls = BertForSequenceClassification.from_pretrained(
    'bert_mlm_adapted', num_labels=NUM_CLASSES, ignore_mismatched_sizes=True)

acc_mlm, mlm_cls_losses, mlm_cls_accs, mlm_true, mlm_preds = bert_train_eval(
    bert_mlm_cls, train_loader, test_loader, BERT_EPOCHS, BERT_LR, 'MLM->CLS',
    wandb_run_name="Part3-MLM-Classification",
    wandb_group="Part3-BERT-MLM",
    extra_config={"phase": "cls_after_mlm", "mlm_pretrained": True},
)

RESULTS['BERT MLM->CLS (Part 3)'] = acc_mlm
print(f'Final MLM->CLS Accuracy: {acc_mlm*100:.2f}%')
print(classification_report(mlm_true, mlm_preds, target_names=data['categories']))

# Plots
fig, axes = plt.subplots(1, 3, figsize=(16,4))
axes[0].plot(range(1,MLM_EPOCHS+1), mlm_losses, marker='o', color='#88c0d0', linewidth=2)
axes[0].set_title('Phase 1 — MLM Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True, linestyle='--', alpha=0.5)
axes[1].plot(range(1,BERT_EPOCHS+1), mlm_cls_losses, marker='o', color='#5e81ac', linewidth=2)
axes[1].set_title('Phase 2 — CLS Loss'); axes[1].set_xlabel('Epoch'); axes[1].grid(True, linestyle='--', alpha=0.5)
bars = axes[2].bar(['BERT direct\n(Part 2)','BERT MLM->CLS\n(Part 3)'],
                   [acc_bert, acc_mlm], color=['#81a1c1','#bf616a'], edgecolor='black', width=0.4)
for bar, acc in zip(bars, [acc_bert, acc_mlm]):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{acc*100:.2f}%', ha='center', va='bottom', fontsize=11)
axes[2].set_ylim(0,1); axes[2].set_title('Part 2 vs Part 3'); axes[2].grid(axis='y', linestyle='--', alpha=0.5)
plt.suptitle('Part 3 — MLM Domain Adaptation', fontsize=13)
plt.tight_layout(); plt.savefig('part3_mlm.png', dpi=150); plt.show(); plt.close()

# Upload Part 3 comparison chart to W&B
with wandb.init(project=WANDB_PROJECT, name="Part3-Comparison",
                group="Part3-BERT-MLM", config=WANDB_CONFIG, reinit=True):
    wandb.log({"part3_comparison": wandb.Image("part3_mlm.png",
               caption="Part 3: MLM loss, CLS loss, Part2 vs Part3")})

bert_mlm_cls.save_pretrained('bert_mlm_cls_checkpoint')
print('Saved: bert_mlm_cls_checkpoint/ | Part 3 complete.')


MLM: 3000 docs | 188 batches/epoch


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 7301.99it/s]
BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [MLM] Epoch 1 Step 50/188 Loss: 2.6354
  [MLM] Epoch 1 Step 100/188 Loss: 2.0295
  [MLM] Epoch 1 Step 150/188 Loss: 1.8736

  [MLM] Epoch 1 avg loss: 2.5397



mlm/epoch_loss,▁
mlm/lr,▃▄▅▅▆█▇▇▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁
mlm/step_loss,▆▇▅▅▅▅▆▂█▃▆▄▄▆▆▆▃▅▅▅▃▂▅▄▃▄▂▃▅▅▅█▁▄▄▃▄▂▆▄
mlm_epoch,▁
mlm_step,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█
mlm/epoch_loss,2.53967
mlm/lr,0
mlm/step_loss,2.10321
mlm_epoch,1
mlm_step,188


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.08it/s]


...

  [MLM->CLS] Epoch 1 Step 50/188 Loss: 2.8232
  [MLM->CLS] Epoch 1 Step 100/188 Loss: 1.9119
  [MLM->CLS] Epoch 1 Step 150/188 Loss: 2.0127

  [MLM->CLS] Epoch 1/3 — Loss: 2.3227  Acc: 58.22%

  [MLM->CLS] Epoch 2 Step 50/188 Loss: 1.2124
  [MLM->CLS] Epoch 2 Step 100/188 Loss: 1.3370
  [MLM->CLS] Epoch 2 Step 150/188 Loss: 1.4428

  [MLM->CLS] Epoch 2/3 — Loss: 1.3253  Acc: 65.31%

  [MLM->CLS] Epoch 3 Step 50/188 Loss: 0.9881
  [MLM->CLS] Epoch 3 Step 100/188 Loss: 1.2188
  [MLM->CLS] Epoch 3 Step 150/188 Loss: 0.8180

  [MLM->CLS] Epoch 3/3 — Loss: 1.0155  Acc: 66.21%



epoch,▁▅█
eval/accuracy,▁▇█
step,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇██
train/epoch_loss,█▃▁
train/lr,▁▄▄▆▇████▇▇▇▇▇▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▂▂▁▁▁▁▁
train/step_loss,██▇▇▆▆▆▆▅▅▅▅▅▄▅▄▄▃▄▂▄▃▃▃▃▂▃▃▃▃▂▃▃▃▃▁▂▃▂▂
epoch,3
eval/accuracy,0.66207
final_test_accuracy,0.66207
step,564
train/epoch_loss,1.01551


Final MLM->CLS Accuracy: 66.21%
                          precision    recall  f1-score   support

             alt.atheism       0.39      0.30      0.34       160
           comp.graphics       0.69      0.64      0.66       195
 comp.os.ms-windows.misc       0.64      0.58      0.61       197
comp.sys.ibm.pc.hardware       0.54      0.35      0.42       196
   comp.sys.mac.hardware       0.53      0.67      0.59       193
          comp.windows.x       0.77      0.81      0.79       198
            misc.forsale       0.81      0.74      0.78       195
               rec.autos       0.48      0.78      0.60       198
         rec.motorcycles       0.70      0.62      0.66       199
      rec.sport.baseball       0.91      0.79      0.85       199
        rec.sport.hockey       0.88      0.89      0.88       200
               sci.crypt       0.70      0.70      0.70       198
         sci.electronics       0.64      0.70      0.67       197
                 sci.med       0.82      0.

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]

Saved: bert_mlm_cls_checkpoint/ | Part 3 complete.


---
## Part 4 — Llama-3.2-1B Zero-Shot & Few-Shot
No weight updates — purely in-context learning.

**Why CPU for Llama?** After Parts 2 & 3, BERT holds ~9 GB of MPS memory. Running the k=5 attention (100-example prompt) on top of that → OOM crash. CPU inference at 1-2 s/doc is fast enough for 60 documents.

**Prerequisites:** `huggingface-cli login` (model is gated — free approval)

In [27]:
import gc

free_memory()

categories   = data['categories']
CATEGORY_STR = '\n'.join(f'  {c}' for c in categories)

# Build evaluation subset: EVAL_SIZE // 20 docs per class from test set
eval_pos = []
for cls_id in range(NUM_CLASSES):
    positions = [p for p, l in enumerate(data['test_labels']) if l == cls_id]
    n = max(1, EVAL_SIZE // NUM_CLASSES)
    eval_pos.extend(random.sample(positions, min(n, len(positions))))

eval_texts  = [data['cleaned_llm'][data['test_idx'][p]][:300] for p in eval_pos]
eval_labels = [data['test_labels'][p]                         for p in eval_pos]
train_llm   = [data['cleaned_llm'][i][:FEW_SHOT_EX_CHARS]    for i in data['train_idx']]
print(f'Eval set: {len(eval_texts)} docs ({len(eval_texts)//NUM_CLASSES} per class)')

print(f'Loading {LLAMA_MODEL} on CPU in float16...')
llama_tok = AutoTokenizer.from_pretrained(LLAMA_MODEL)
if llama_tok.pad_token is None:
    llama_tok.pad_token = llama_tok.eos_token

llama_mdl = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL, torch_dtype=torch.float16, device_map='cpu', low_cpu_mem_usage=True)
llama_mdl.eval()
print('Model loaded on CPU.')

llama_pipe = hf_pipeline(
    'text-generation', model=llama_mdl, tokenizer=llama_tok,
    max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
    return_full_text=False, pad_token_id=llama_tok.eos_token_id,
)

def zero_shot_msgs(doc):
    return [
        {'role': 'system',
         'content': ('Classify the article into exactly ONE of the 20 categories.\n'
                     'Reply with ONLY the exact category name. No explanation.\n\n'
                     f'Categories:\n{CATEGORY_STR}')},
        {'role': 'user', 'content': f'Article:\n{doc}\n\nCategory:'},
    ]

def few_shot_msgs(doc, examples):
    demo = ''.join(f'Article: {t}\nCategory: {l}\n\n' for t, l in examples)
    return [
        {'role': 'system',
         'content': ('Classify the article into exactly ONE of the 20 categories.\n'
                     'Reply with ONLY the exact category name. No explanation.\n\n'
                     f'Categories:\n{CATEGORY_STR}')},
        {'role': 'user', 'content': f'Examples:\n\n{demo}Now classify:\n{doc}\n\nCategory:'},
    ]

def to_prompt(msgs):
    return llama_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse_output(raw):
    line = raw.strip().split('\n')[0].strip().lower()
    for cat in categories:
        if line == cat.lower(): return cat
    for cat in categories:
        if cat.lower() in line: return cat
    return None

def run_eval(prompt_fn, label, k_shot=0):
    """Run Llama inference and log per-doc results + final accuracy to W&B."""
    preds, trues, bad = [], [], 0

    # Open a W&B run for this specific k-shot experiment
    run = wandb.init(
        project=WANDB_PROJECT,
        name=f"Part4-Llama-k{k_shot}",
        group="Part4-Llama",
        config={**WANDB_CONFIG,
                "llama_k_shot":     k_shot,
                "llama_eval_size":  len(eval_texts),
                "max_new_tokens":   MAX_NEW_TOKENS},
        reinit=True,
    )

    # W&B Table: one row per document — lets you inspect individual predictions
    prediction_table = wandb.Table(columns=["doc_id", "true_label", "predicted_label", "correct"])

    for i, (doc, lbl) in enumerate(zip(eval_texts, eval_labels)):
        print(f'  [{label}] {i+1}/{len(eval_texts)}', end='\r', flush=True)
        try:
            out  = llama_pipe(to_prompt(prompt_fn(doc)))[0]['generated_text']
            pred = parse_output(out)
        except Exception:
            pred = None

        if pred is None:
            bad += 1; pred_idx = -1; pred_name = "UNPARSEABLE"
        else:
            pred_idx = categories.index(pred); pred_name = pred

        preds.append(pred_idx)
        trues.append(lbl)

        # Log each prediction to the W&B table
        prediction_table.add_data(i, categories[lbl], pred_name, pred_name == categories[lbl])

    print()
    acc = accuracy_score(trues, preds)
    print(f'  [{label}] Accuracy: {acc*100:.2f}%  (unparseable: {bad}/{len(eval_texts)})')

    wandb.log({
        "llama/accuracy":           acc,
        "llama/unparseable_count":  bad,
        "llama/k_shot":             k_shot,
        "llama/predictions":        prediction_table,
        #"llama/confusion_matrix":   wandb.plot.confusion_matrix(
            #probs=None,
            #y_true=trues,
            #preds=preds,
            #class_names=list(categories),
        #),
    })
    wandb.summary["accuracy"] = acc
    run.finish()
    return acc


# Zero-shot
print(f'\nZero-shot ({len(eval_texts)} docs)...')
acc_zero = run_eval(zero_shot_msgs, 'Zero-shot', k_shot=0)
RESULTS['Llama-3.2-1B Zero-shot'] = acc_zero

# Few-shot: k=1,3,5
def sample_few_shot(k):
    examples = []
    for cls_id, cls_name in enumerate(categories):
        cls_idx = [i for i, l in enumerate(data['train_labels']) if l == cls_id]
        for i in random.sample(cls_idx, min(k, len(cls_idx))):
            examples.append((train_llm[i], cls_name))
    random.shuffle(examples)
    return examples

few_shot_accs = {}
for k in FEW_SHOT_KS:
    print(f'\nFew-shot k={k} ({k*NUM_CLASSES} examples in prompt)...')
    examples = sample_few_shot(k)
    acc_k = run_eval(lambda doc, ex=examples: few_shot_msgs(doc, ex), f'k={k}', k_shot=k)
    few_shot_accs[k] = acc_k
    RESULTS[f'Llama-3.2-1B k={k}'] = acc_k
    free_memory()

# Learning curve
ks   = [0] + list(few_shot_accs.keys())
accs = [acc_zero] + list(few_shot_accs.values())
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(ks, [a*100 for a in accs], marker='o', linewidth=2.5, color='#5e81ac', markersize=10)
for k, a in zip(ks, accs):
    ax.annotate(f'{a*100:.1f}%', (k, a*100), textcoords='offset points',
                xytext=(0,10), ha='center', fontsize=10)
ax.set_xticks(ks)
ax.set_xticklabels(['0\n(zero-shot)'] + [f'k={k}' for k in few_shot_accs])
ax.set_xlabel('Examples per class in prompt', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Part 4 — Llama-3.2-1B Zero-Shot vs Few-Shot (CPU)', fontsize=13)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout(); plt.savefig('part4_llama.png', dpi=150); plt.show(); plt.close()

# Log the learning curve and a summary bar to W&B
with wandb.init(project=WANDB_PROJECT, name="Part4-Llama-Summary",
                group="Part4-Llama", config=WANDB_CONFIG, reinit=True):
    wandb.log({
        "part4_learning_curve": wandb.Image("part4_llama.png",
                                            caption="Llama zero-shot vs few-shot accuracy"),
        "part4_k_vs_accuracy":  wandb.plot.line(
            wandb.Table(columns=["k", "accuracy"],
                        data=[[k, round(a*100,2)] for k,a in zip(ks,accs)]),
            "k", "accuracy", title="Llama: k-shot vs Accuracy"
        ),
    })

del llama_mdl, llama_pipe; free_memory()
print('Part 4 complete.')


Eval set: 60 docs (3 per class)
Loading meta-llama/Llama-3.2-1B-Instruct on CPU in float16...


Loading weights: 100%|██████████| 146/146 [00:05<00:00, 28.29it/s]


Model loaded on CPU.

Zero-shot (60 docs)...


Both `max_new_tokens` (=15) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=15) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=15) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  [k=5] Accuracy: 5.00%  (unparseable: 55/60)


llama/accuracy,▁
llama/k_shot,▁
llama/unparseable_count,▁
accuracy,0.05
llama/accuracy,0.05
llama/k_shot,5
llama/unparseable_count,55


wandb-core(92350) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(92351) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Part 4 complete.


---
## Final Comparison

In [28]:
def part_color(name):
    if any(x in name for x in ['W2V','TF-IDF','FastText']): return '#5e81ac'
    if 'Part 2' in name: return '#bf616a'
    if 'Part 3' in name: return '#d08770'
    return '#b48ead'

names = list(RESULTS.keys())
accs  = list(RESULTS.values())

fig, ax = plt.subplots(figsize=(max(14, len(names)*1.6), 6))
bars = ax.bar(names, accs, color=[part_color(n) for n in names], edgecolor='black', width=0.65)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{acc*100:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylim(0,1); ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('NALAPRO — Full Comparison (Parts 1-4)', fontsize=13)
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(handles=[
    mpatches.Patch(facecolor='#5e81ac', label='Part 1 — Classical'),
    mpatches.Patch(facecolor='#bf616a', label='Part 2 — BERT direct'),
    mpatches.Patch(facecolor='#d08770', label='Part 3 — BERT MLM->CLS'),
    mpatches.Patch(facecolor='#b48ead', label='Part 4 — Llama zero/few-shot'),
], loc='lower right', fontsize=9)
plt.tight_layout(); plt.savefig('full_comparison.png', dpi=150); plt.show(); plt.close()

print(f'{"Method":<35} {"Accuracy":>10}')
print('-'*48)
for name, acc in sorted(RESULTS.items(), key=lambda x: -x[1]):
    print(f'{name:<35} {acc*100:>9.2f}%')

with open('results.json', 'w') as f:
    json.dump({k: round(v*100, 2) for k, v in RESULTS.items()}, f, indent=2)

# ── Final W&B run: master comparison across all Parts ─────────────────────────
# This single run summarises every method in one place.
# The W&B link from this run is the one to paste into your report/code header.
with wandb.init(project=WANDB_PROJECT, name="Final-All-Methods-Comparison",
                group="FinalSummary", config=WANDB_CONFIG, reinit=True) as final_run:

    # Full comparison chart as an image
    wandb.log({"final_comparison_chart": wandb.Image("full_comparison.png",
               caption="All methods compared — NALAPRO")})

    # Sortable bar chart in W&B dashboard
    wandb.log({
        "all_methods_accuracy": wandb.plot.bar(
            wandb.Table(
                columns=["Method", "Accuracy (%)"],
                data=[[n, round(a*100, 2)] for n, a in RESULTS.items()]
            ),
            "Method", "Accuracy (%)",
            title="NALAPRO — All Methods Accuracy"
        )
    })

    # Every result as a W&B summary scalar — shows up in the run overview table
    for method, acc in RESULTS.items():
        safe_key = method.replace(" ", "_").replace("(", "").replace(")", "").replace("->", "_")
        wandb.summary[safe_key] = round(acc * 100, 2)

    print(f'\n✅ W&B project URL: {final_run.get_project_url()}')
    print(f'✅ This run URL:    {final_run.get_url()}')
    print('   → Paste the project URL into your code header and report.')

print('\nSaved: full_comparison.png + results.json')


Method                                Accuracy
------------------------------------------------
TF-IDF                                  74.99%
BERT MLM->CLS (Part 3)                  66.21%
BERT (Part 2)                           65.46%
W2V 15ep                                57.93%
FastText                                54.75%
W2V 1ep                                 24.64%
Llama-3.2-1B k=3                        15.00%
Llama-3.2-1B k=1                        11.67%
Llama-3.2-1B Zero-shot                  10.00%
Llama-3.2-1B k=5                         5.00%


wandb-core(92407) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(92408) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


wandb: WARNING The get_project_url method is deprecated and will be removed in a future release. Please use `run.project_url` instead.
wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.



✅ W&B project URL: https://wandb.ai/talhamch-hslu/nalapro-20newsgroups
✅ This run URL:    https://wandb.ai/talhamch-hslu/nalapro-20newsgroups/runs/uxxqvjz6
   → Paste the project URL into your code header and report.


BERT_MLM_CLS_Part_3,66.21
BERT_Part_2,65.46
FastText,54.75
Llama-3.2-1B_Zero-shot,10
Llama-3.2-1B_k=1,11.67
Llama-3.2-1B_k=3,15
Llama-3.2-1B_k=5,5
TF-IDF,74.99
W2V_15ep,57.93
W2V_1ep,24.64



Saved: full_comparison.png + results.json


---
## Discussion

| Finding | Explanation |
|---------|-------------|
| TF-IDF > Word2Vec (mean-pool) | TF-IDF weights rare discriminative words highly. Mean-pooling dilutes them with frequent noise words. |
| More W2V epochs → tighter clusters | Semantically related words move closer in vector space as co-occurrence statistics converge. |
| FastText ≈ W2V 15ep | Subword advantage is modest for clean English; bigger gains on noisy or morphologically rich text. |
| BERT >> all Part 1 | Contextual representations capture polysemy and long-range dependencies that static embeddings cannot. |
| MLM adaptation → marginal improvement | One MLM epoch shifts BERT's vocabulary statistics toward newsgroup jargon before the labelled task. |
| Llama zero-shot < fine-tuned BERT | No weight updates; category names like `alt.atheism` are not self-explanatory. |
| Few-shot improves with k | Demonstrations reduce ambiguity, but gains plateau as prompt length approaches the context limit. |